# Section 5 — Encoding a Node Test

This notebook tests the implementation of **Section 5**:
1. **Definition 4 (Less-or-equal comparator)**: Recursive formula $\psi_{b_f - 1}$ comparing encoded integer $X_f$ against fixed threshold index $\tau$ in linear size without quadratic expansion.
2. **Proposition 2 (Linearity and zero prefix)**: Ensuring the high-order zero prefix forces bits above $\tau$ to 0 automatically.
3. **OBDD construction**: Building $\text{nt}(f, \tau) := \psi_{b_f - 1}$ in `dd.autoref.BDD` and verifying exact agreement with both discrete $X_f \le \tau$ and original continuous $x_f \le \text{threshold}$ tests.
4. **Partial assignment (projection $\rho$)**: Verifying that passing $\rho$ simplifies the comparator on the fly (for Section 12).

In [1]:
import os
import sys
import numpy as np
import dd.autoref as _bdd

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from rfbdd.forest import Tree, Forest
from rfbdd.discretize import discretize
from rfbdd.comparator import encode_node_test, eval_node_test_formula

def check(condition: bool, message: str) -> None:
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {message}")
    assert condition, message

print("Setup complete.")

Setup complete.


## 5.1 Exhaustive Verification of Definition 4 Recurrence

> **Definition 4**: *$\psi_{-1} = \top$, and for $0 \le j \le b_f - 1$:*
> $$\psi_j = \begin{cases} \neg B_f^j \wedge \psi_{j-1} & \text{if } \tau_j = 0 \\ \neg B_f^j \vee \psi_{j-1} & \text{if } \tau_j = 1 \end{cases}$$

We test every possible bit combination of width $b_f \in \{1, 2, 3, 4\}$ and every threshold $\tau \in \{0, \dots, 2^{b_f}-1\}$ against the numerical condition $X_f \le \tau$.

In [2]:
all_passed = True
for bf in [1, 2, 3, 4]:
    max_val = 1 << bf
    for tau in range(max_val):
        for x_int in range(max_val):
            bits = [bool((x_int >> j) & 1) for j in range(bf)]
            formula_res = eval_node_test_formula(bits, tau)
            ground_truth = (x_int <= tau)
            if formula_res != ground_truth:
                all_passed = False
                break

check(all_passed, "Definition 4 formula matches (X_f <= tau) exhaustively for all bf in 1..4 and all tau")

[PASS] Definition 4 formula matches (X_f <= tau) exhaustively for all bf in 1..4 and all tau


## 5.2 OBDD Construction for Forest Node Tests

We instantiate the sample forest from Sections 3–4 and construct OBDDs for each internal node test $\text{nt}(f, \tau) := \psi_{b_f - 1}$.

In [3]:
t0 = Tree(
    feature=np.array([0, -1, 1, -1, -1], dtype=np.int64),
    threshold=np.array([5.7, np.nan, 10.0, np.nan, np.nan], dtype=np.float64),
    left=np.array([1, -1, 3, -1, -1], dtype=np.int64),
    right=np.array([2, -1, 4, -1, -1], dtype=np.int64),
    label=np.array([-1, 0, -1, 0, 1], dtype=np.int64),
)
t1 = Tree(
    feature=np.array([0, -1, 0, -1, -1], dtype=np.int64),
    threshold=np.array([2.5, np.nan, 8.1, np.nan, np.nan], dtype=np.float64),
    left=np.array([1, -1, 3, -1, -1], dtype=np.int64),
    right=np.array([2, -1, 4, -1, -1], dtype=np.int64),
    label=np.array([-1, 0, -1, 1, 0], dtype=np.int64),
)
forest = Forest(trees=[t0, t1], n_features=2, n_classes=2, classes=np.array([0, 1]), name="toy_forest")
disc = discretize(forest)

bdd = _bdd.BDD()
for var_name in disc.all_bit_names:
    bdd.declare(var_name)

# Test node 0 of tree 0: feature 0, threshold 5.7, cell index tau = 1
tau0 = disc.taus[0][0]
nt0 = encode_node_test(bdd, disc, f=0, tau=tau0)
print(f"Node 0 test BDD created with tau={tau0}")

Node 0 test BDD created with tau=1


## 5.3 Truth Table and Proposition 1 Equivalence

We evaluate the node test BDD on all $2^3 = 8$ input bit assignments and verify that `bdd.let(assign, nt0) == bdd.true` if and only if $\text{cell}[0] \le \tau_0$.

In [4]:
all_correct = True
for b0 in [False, True]:
    for b1 in [False, True]:
        for b2 in [False, True]:
            assign = {"x0_0": b0, "x0_1": b1, "x1_0": b2}
            cells = disc.decode_cells(assign)
            expected = (cells[0] <= tau0)
            val_bdd = bdd.let(assign, nt0)
            actual = (val_bdd == bdd.true)
            if actual != expected:
                all_correct = False

check(all_correct, f"Node test nt(0, {tau0}) BDD evaluated correctly on all 8 assignments")

[PASS] Node test nt(0, 1) BDD evaluated correctly on all 8 assignments


## 5.4 Partial Assignment Projection ($\rho$)

For the *Budgeted Frontier* of Section 12, a cube $\rho$ fixes a subset of input bits. We test that passing $\rho$ to `encode_node_test` simplifies the formula directly at build time.

In [5]:
# Fix high-order bit x0_1 = True (meaning cell index is at least 2)
# Since tau = 1, fixing x0_1 = True must make nt(0, 1) collapse to bdd.false
rho_false = {"x0_1": True}
nt_projected_false = encode_node_test(bdd, disc, f=0, tau=tau0, rho=rho_false)
check(nt_projected_false == bdd.false, "Projected node test collapsed to FALSE when x0_1 = True")

# Fix high-order bit x0_1 = False and x0_0 = False (cell index 0)
# Since 0 <= 1, it must collapse to bdd.true
rho_true = {"x0_1": False, "x0_0": False}
nt_projected_true = encode_node_test(bdd, disc, f=0, tau=tau0, rho=rho_true)
check(nt_projected_true == bdd.true, "Projected node test collapsed to TRUE when x0_1 = False, x0_0 = False")

[PASS] Projected node test collapsed to FALSE when x0_1 = True
[PASS] Projected node test collapsed to TRUE when x0_1 = False, x0_0 = False
